In [0]:
from pyspark.sql.functions import (
    col, lit, trim, upper, when, concat, monotonically_increasing_id,
    date_format, year, quarter, month, dayofmonth, dayofweek, 
    row_number, coalesce, min as spark_min, max as spark_max
)
from pyspark.sql.window import Window

chicago = spark.table("workspace.default.silver_chicago_inspections")
dallas = spark.table("workspace.default.silver_dallas_inspections")

print(f"Chicago Silver: {chicago.count()} rows")
print(f"Dallas Silver: {dallas.count()} rows")

In [0]:
# ============================================================================
# dim_date — Standard date dimension generated from all inspection dates
# ============================================================================
from pyspark.sql.functions import expr, concat_ws

all_dates = (
    chicago.select("inspection_date")
    .union(dallas.select("inspection_date"))
    .where(col("inspection_date").isNotNull())
    .distinct()
)

dim_date = (
    all_dates
    .withColumn("date_key", date_format("inspection_date", "yyyyMMdd").cast("int"))
    .withColumn("full_date", col("inspection_date"))
    .withColumn("year", year("inspection_date"))
    .withColumn("quarter", quarter("inspection_date"))
    .withColumn("month", month("inspection_date"))
    .withColumn("month_name", date_format("inspection_date", "MMMM"))
    .withColumn("day", dayofmonth("inspection_date"))
    .withColumn("day_of_week", date_format("inspection_date", "EEEE"))
    .withColumn("day_of_week_num", dayofweek("inspection_date"))
    .withColumn("is_weekend", dayofweek("inspection_date").isin(1, 7))
    .withColumn("fiscal_year", 
        when(month("inspection_date") >= 10,
             concat(lit("FY"), (year("inspection_date") + 1).cast("string")))
        .otherwise(concat(lit("FY"), year("inspection_date").cast("string"))))
    .select("date_key", "full_date", "year", "quarter", "month", "month_name",
            "day", "day_of_week", "day_of_week_num", "is_weekend", "fiscal_year")
    .distinct()
    .orderBy("date_key")
)

dim_date.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_date")
print(f"dim_date: {spark.table('workspace.default.dim_date').count()} rows")
display(dim_date.limit(5))

In [0]:
# ============================================================================
# dim_establishment — Business/restaurant info from both cities
# NK: license (Chicago) or name|address composite (Dallas)
# ============================================================================

chi_establishments = (
    chicago
    .select(
        col("license_number").alias("establishment_id"),
        lit("Chicago").alias("source_city"),
        col("dba_name"),
        col("aka_name"),
        col("license_number"),
        col("facility_type"),
        col("risk_level")
    )
    .where(col("establishment_id").isNotNull())
    .distinct()
)

dal_establishments = (
    dallas
    .withColumn("establishment_id", concat(col("dba_name"), lit("|"), col("address")))
    .select(
        col("establishment_id"),
        lit("Dallas").alias("source_city"),
        col("dba_name"),
        lit(None).cast("string").alias("aka_name"),
        lit(None).cast("string").alias("license_number"),
        lit("Restaurant").alias("facility_type"),
        lit(None).cast("string").alias("risk_level")
    )
    .distinct()
)

dim_establishment = (
    chi_establishments.union(dal_establishments)
    .dropDuplicates(["establishment_id", "source_city"])
    .withColumn("establishment_key", monotonically_increasing_id())
    .select("establishment_key", "establishment_id", "source_city", "dba_name",
            "aka_name", "license_number", "facility_type", "risk_level")
)

dim_establishment.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_establishment")
print(f"dim_establishment: {spark.table('workspace.default.dim_establishment').count()} rows")
display(dim_establishment.limit(5))

In [0]:
# ============================================================================
# dim_violation is managed by DLT pipeline (dim_violation_scd2)
# We just need to add a surrogate key for the fact table join
# ============================================================================

scd2 = spark.table("workspace.default.dim_violation_scd2")
print(f"dim_violation_scd2: {scd2.count()} rows")
print(f"Current records: {scd2.where(col('__END_AT').isNull()).count()}")
display(scd2.limit(5))

In [0]:
# ============================================================================
# dim_location — Geographic info from both cities
# NK: address|zip composite
# ============================================================================

all_locations = (
    chicago.select("address", "city", "state", "zip_code", "latitude", "longitude")
    .union(dallas.select("address", "city", "state", "zip_code", "latitude", "longitude"))
    .where(col("address").isNotNull())
    .withColumn("location_id", concat(col("address"), lit("|"), col("zip_code")))
    .dropDuplicates(["location_id"])
    .withColumn("location_key", monotonically_increasing_id())
    .select("location_key", "location_id", "address", "city", "state", 
            "zip_code", "latitude", "longitude")
)

all_locations.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_location")
print(f"dim_location: {spark.table('workspace.default.dim_location').count()} rows")
display(all_locations.limit(5))

In [0]:
# ============================================================================
# dim_inspection_type — Lookup for inspection types from both cities
# ============================================================================

all_types = (
    chicago.select("source_city", "inspection_type")
    .union(dallas.select("source_city", "inspection_type"))
    .distinct()
    .withColumn("inspection_type_key", monotonically_increasing_id())
    .withColumn("inspection_category",
        when(col("inspection_type").rlike("(?i)(canvass|license|routine)"), "Routine")
        .when(col("inspection_type").rlike("(?i)(re-inspection|follow)"), "Re-Inspection")
        .when(col("inspection_type").rlike("(?i)complaint"), "Complaint")
        .otherwise("Other"))
    .select("inspection_type_key", "inspection_type", "source_city", "inspection_category")
)

all_types.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_inspection_type")
print(f"dim_inspection_type: {spark.table('workspace.default.dim_inspection_type').count()} rows")
display(all_types)

In [0]:
# ============================================================================
# dim_inspection_result — Result/score mapping
# ============================================================================

all_results = (
    chicago.select("results").distinct()
    .union(dallas.select("results").distinct())
    .distinct()
    .withColumn("result_key", monotonically_increasing_id())
    .withColumn("derived_score",
        when(col("results") == "Pass", 90)
        .when(col("results") == "Pass w/ Conditions", 80)
        .when(col("results") == "Fail", 70)
        .when(col("results") == "No Entry", 0)
        .otherwise(lit(None).cast("int")))
    .withColumn("result_category",
        when(col("results").isin("Pass", "Pass w/ Conditions"), "Pass")
        .when(col("results") == "Fail", "Fail")
        .otherwise("Other"))
    .withColumnRenamed("results", "result_text")
    .select("result_key", "result_text", "derived_score", "result_category")
)

all_results.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_inspection_result")
print(f"dim_inspection_result: {spark.table('workspace.default.dim_inspection_result').count()} rows")
display(all_results)

In [0]:
# ============================================================================
# fact_inspection_violation — Join Silver data to all dimension keys
# ============================================================================

combined = chicago.union(dallas)

d_date = spark.table("workspace.default.dim_date")
d_est = spark.table("workspace.default.dim_establishment")
d_loc = spark.table("workspace.default.dim_location")
d_type = spark.table("workspace.default.dim_inspection_type")
d_result = spark.table("workspace.default.dim_inspection_result")

# dim_violation_scd2 — no surrogate key, so we generate one
# Only join to current records (__END_AT IS NULL)
d_viol = (
    spark.table("workspace.default.dim_violation_scd2")
    .where(col("__END_AT").isNull())
    .withColumn("violation_key", monotonically_increasing_id())
)

combined_keyed = (
    combined
    .withColumn("date_key", date_format("inspection_date", "yyyyMMdd").cast("int"))
    .withColumn("establishment_id",
        when(col("source_city") == "Chicago", col("license_number"))
        .otherwise(concat(col("dba_name"), lit("|"), col("address"))))
    .withColumn("location_id", concat(col("address"), lit("|"), col("zip_code")))
)

fact = (
    combined_keyed
    .join(d_date.select("date_key"), on="date_key", how="left")
    .join(
        d_est.select(col("establishment_key"), col("establishment_id").alias("est_id"), col("source_city").alias("est_city")),
        on=(combined_keyed["establishment_id"] == col("est_id")) & (combined_keyed["source_city"] == col("est_city")),
        how="left"
    ).drop("est_id", "est_city")
    .join(
        d_loc.select(col("location_key"), col("location_id").alias("loc_id")),
        on=combined_keyed["location_id"] == col("loc_id"),
        how="left"
    ).drop("loc_id")
    # Join to SCD2 violation dimension (current records only, with generated key)
    .join(
        d_viol.select(col("violation_key"), col("violation_code").alias("v_code"), col("source_city").alias("v_city"), col("violation_description").alias("v_desc")),
        on=(combined_keyed["violation_code"] == col("v_code")) & 
           (combined_keyed["source_city"] == col("v_city")) & 
           (combined_keyed["violation_description"] == col("v_desc")),
        how="left"
    ).drop("v_code", "v_city", "v_desc")
    .join(
        d_type.select(col("inspection_type_key"), col("inspection_type").alias("t_type"), col("source_city").alias("t_city")),
        on=(combined_keyed["inspection_type"] == col("t_type")) & (combined_keyed["source_city"] == col("t_city")),
        how="left"
    ).drop("t_type", "t_city")
    .join(
        d_result.select(col("result_key"), col("result_text")),
        on=combined_keyed["results"] == col("result_text"),
        how="left"
    ).drop("result_text")
    .withColumn("inspection_violation_id", monotonically_increasing_id())
    .select(
        "inspection_violation_id",
        "date_key",
        "establishment_key",
        "location_key",
        "violation_key",
        "inspection_type_key",
        "result_key",
        "source_city",
        "inspection_id",
        "inspection_score",
        "violation_points",
        "violation_slot"
    )
)

fact.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.fact_inspection_violation")
print(f"fact_inspection_violation: {spark.table('workspace.default.fact_inspection_violation').count()} rows")

In [0]:
fact = spark.table("workspace.default.fact_inspection_violation")

print("=== GOLD LAYER SUMMARY ===")
print(f"fact_inspection_violation: {fact.count()} rows")
print(f"dim_date: {spark.table('workspace.default.dim_date').count()} rows")
print(f"dim_establishment: {spark.table('workspace.default.dim_establishment').count()} rows")
print(f"dim_location: {spark.table('workspace.default.dim_location').count()} rows")
print(f"dim_violation_scd2: {spark.table('workspace.default.dim_violation_scd2').count()} rows")
print(f"dim_inspection_type: {spark.table('workspace.default.dim_inspection_type').count()} rows")
print(f"dim_inspection_result: {spark.table('workspace.default.dim_inspection_result').count()} rows")

print("\n=== FACT TABLE FK NULL CHECK ===")
for fk in ["date_key", "establishment_key", "location_key", "violation_key", "inspection_type_key", "result_key"]:
    nulls = fact.where(col(fk).isNull()).count()
    print(f"  {fk}: {nulls} nulls {'✓' if nulls == 0 else '⚠ has nulls'}")

print("\n=== FACT BY SOURCE CITY ===")
display(fact.groupBy("source_city").count())